In [0]:
%sql

MERGE INTO laliga.gold.fact_stats AS target
USING(
  WITH sums_stats AS (
    SELECT 
      IdJugador,
      IdEquipo,
      id_partido,
      CASE WHEN CincoInicial = 1 THEN 'Titular' ELSE 'Suplente' END AS cinco_inicial,
      SUM(TiempoJuego) AS TiempoJuego,
      SUM(Puntos) AS Puntos,
      SUM(tiros_2p_aciertos) AS tiros_2_aciertos,
      SUM(tiros_2p_fallados) AS tiros_2_fallos,
      SUM(tiros_2p_totales) AS tiros_2_totales,
      SUM(tiros_3p_aciertos) AS tiros_3_aciertos,
      SUM(tiros_3p_fallados) AS tiros_3_fallos,
      SUM(tiros_3p_totales) AS tiros_3_totales,
      SUM(libres_aciertos) AS tiros_libres_aciertos,
      SUM(libres_fallados) AS tiros_libres_fallos,
      SUM(libres_totales) AS tiros_libres_totales,
      SUM(ReboteDefensivo) AS rebote_def,
      SUM(ReboteOfensivo) AS rebote_ofensivo,
      SUM(RebotesTotales) AS rebs_totales, 
      SUM(Asistencias) AS asist, 
      SUM(Recuperaciones) AS recups,
      SUM(Perdidas) AS perdidas,
      SUM(TaponCometido) AS tapas_realizadas, 
      SUM(TaponRecibido) AS tapas_recibidas,
      SUM(FaltaCometida) AS faltas_realizadas, 
      SUM(FaltaRecibida) AS faltas_recibidas 
    FROM laliga.silver.players_stats
    GROUP BY IdEquipo, IdJugador, id_partido, cinco_inicial
  ),
  metricas AS (
    SELECT
      *,
      ROUND((tiros_2_aciertos * 100 / NULLIF(tiros_2_totales, 0)), 2) AS pctg_2,
      ROUND(tiros_3_aciertos * 100 / NULLIF(tiros_3_totales, 0), 2) AS pctg_3,
      ROUND(tiros_libres_aciertos * 100 / NULLIF(tiros_libres_totales, 0), 2) AS pctg_tiro_libre,
      ROUND(Puntos / NULLIF(2 * (tiros_2_totales + tiros_3_totales + 0.44 * tiros_libres_totales), 0), 3) AS ts,
      ROUND((tiros_3_aciertos + 1.5 * tiros_3_aciertos) / NULLIF(tiros_2_totales + tiros_3_totales, 0), 2) AS efg
    FROM sums_stats  
  )
  SELECT 
    m.IdEquipo,
    m.IdJugador,
    m.id_partido,
    cinco_inicial,
    TiempoJuego,
    Puntos,
    tiros_2_aciertos,
    tiros_2_fallos,
    tiros_2_totales,
    pctg_2,
    tiros_3_aciertos,
    tiros_3_fallos,    
    tiros_3_totales, 
    pctg_3, 
    tiros_libres_aciertos,
    tiros_libres_fallos,
    m.tiros_libres_totales,
    pctg_tiro_libre,
    ts,
    efg,
    ROUND((tiros_2_totales + tiros_3_totales + 0.44 * m.tiros_libres_totales + perdidas) / NULLIF(g.posesiones, 0) * 100, 2) AS uso_pctg,
    rebote_def,
    rebote_ofensivo,   
    rebs_totales, 
    asist, 
    recups,
    perdidas,
    tapas_realizadas, 
    tapas_recibidas,  
    faltas_realizadas, 
    faltas_recibidas,
    'laliga.silver.players_stats' AS _source_table,
    CURRENT_TIMESTAMP() AS _processing_timestamp 
  FROM metricas m
  JOIN laliga.gold.fact_partidos_stats g
  ON m.idequipo = g.IdEquipo AND m.id_partido = g.id_partido
) AS source
ON target.IdJugador = source.IdJugador AND target.id_partido = source.id_partido
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT TiempoJuego FROM laliga.gold.fact_stats 
WHERE IdJugador = '172004'
